# MuonClip Angular WeightWatcher — initial / best / final

This is the **canonical** angular spectral-analysis notebook for the one-head NanoGPT MuonClip baseline. It analyzes three actual saved checkpoints from one run:

- `checkpoint_initial.pt` — the saved step-zero model;
- `checkpoint_best.pt` — the best-validation model;
- `checkpoint_final.pt` — the completed final model.

No checkpoint is reconstructed from a seed and no endpoint is silently substituted. The three relative flows are

$$W_0\to W_{best},\qquad W_0\to W_{final},\qquad W_{best}\to W_{final}.$$

For every WeightWatcher matrix and for both gauge-invariant angular sectors (`tilt` and `twist`), the notebook compares the observed flow against a matched Haar/Stiefel random-angular null.


## Scientific contract

For an angular eigenvalue $\lambda$ with bounded support $0\le\lambda\le u$, the continuous projective coordinate is

$$x=\frac{\lambda/u}{1-\lambda/u}.$$

The exact upper endpoint is **not** converted into a giant finite tail value. In the twist sector, $\lambda=4$ corresponds to a $-1$ eigenvalue of the relative orthogonal rotation and can be a discrete reflection/parity atom. Values within `ANGULAR_ENDPOINT_TOL` of the upper endpoint are therefore counted separately and excluded from the continuous projective tail. This prevents a numerical clip such as $1-10^{-12}$ from fabricating an apparent $x\sim10^{12}$ outlier.

The remaining positive continuous values are passed directly to the same package used by WeightWatcher:

```python
powerlaw.Fit(values, discrete=False, verbose=False)
```

We supply **no `xmin` and no `xmax`**. The package chooses the tail start by its MLE/KS procedure. Every continuous observed value from the selected `xmin` through the largest continuous observed value remains in the fit. `ANGULAR_MIN_TAIL` is checked only *after* the package has selected `xmin`; it does not choose the fitting window.

A fitted exponent by itself is not evidence for angular RG structure. For each flow we compare the trained spectrum with random-angular nulls using: $\alpha$, package-selected $x_{min}$, KS $D$, tail population, tail length $\log_{10}(x_{max}/x_{min})$, endpoint-atom counts, a full-continuous-spectrum Monte-Carlo KS test, and a far-tail Monte-Carlo KS test above the trained package-selected $x_{min}$.


## Command-line and Papermill usage

The next cell is tagged `parameters`. Its defaults are read from environment variables, but Papermill can also override them with `-p`. This is deliberate: the `AnalysisConfig` object is constructed in the cell *after* Papermill's injected-parameters cell.

Typical environment-driven run:

```bash
cd /path/to/rg_optimizers
export RG_OPTIMIZERS_ROOT="$PWD"
export RUNROOT=/tmp/<the-same-run-root-used-for-training>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"
export ANGULAR_N_NULL=500
export ANGULAR_ENDPOINT_TOL=1e-10
export ANGULAR_SHOW_PLOTS=0

papermill \
  baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb \
  /tmp/angular_seed${TARGET_SEED}.out.ipynb
```

Papermill parameter override example:

```bash
papermill \
  baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb \
  /tmp/angular_seed777.out.ipynb \
  -p TARGET_SEED 777 \
  -p RUN_DIR /tmp/my-run/results/muon_clip/seed_777 \
  -p ANGULAR_N_NULL 500 \
  -p ANGULAR_SHOW_PLOTS false
```

Optional checkpoint overrides are `INITIAL_CHECKPOINT_PATH`, `BEST_CHECKPOINT_PATH`, and `FINAL_CHECKPOINT_PATH`.


In [ ]:
import os

TARGET_SEED = int(os.environ.get("TARGET_SEED", os.environ.get("RG_SEED", "4242")))
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", os.environ.get("OPTIMIZER_NAME", "muon_clip"))
RUN_DIR = os.environ.get("RUN_DIR", "")
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUNROOT = os.environ.get("RUNROOT", "")
INITIAL_CHECKPOINT_PATH = os.environ.get("INITIAL_CHECKPOINT_PATH", "")
BEST_CHECKPOINT_PATH = os.environ.get("BEST_CHECKPOINT_PATH", "")
FINAL_CHECKPOINT_PATH = os.environ.get("FINAL_CHECKPOINT_PATH", os.environ.get("CHECKPOINT_PATH", ""))
ANGULAR_OUTPUT_DIR = os.environ.get("ANGULAR_OUTPUT_DIR", "")
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "100"))
ANGULAR_N_ENTRY_NULL = int(os.environ.get("ANGULAR_N_ENTRY_NULL", "24"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "1")


In [ ]:
from pathlib import Path
import sys

def _as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() not in {"0", "false", "no", "off"}

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError("Set RG_OPTIMIZERS_ROOT or launch from the rg_optimizers repository")

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

# BEST_CHECKPOINT_PATH and ANGULAR_ENDPOINT_TOL are consumed by the shared
# three-checkpoint resolver. Set them here so environment defaults and
# Papermill -p overrides behave identically.
if _none_if_blank(BEST_CHECKPOINT_PATH):
    os.environ["BEST_CHECKPOINT_PATH"] = str(BEST_CHECKPOINT_PATH)
else:
    os.environ.pop("BEST_CHECKPOINT_PATH", None)
os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)

from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_three_checkpoint import run_three_checkpoint_analysis

CONFIG = AnalysisConfig(
    seed=int(TARGET_SEED),
    optimizer=str(TARGET_OPTIMIZER).lower(),
    run_dir=_none_if_blank(RUN_DIR),
    results_root=_none_if_blank(RESULTS_ROOT),
    runroot=_none_if_blank(RUNROOT),
    initial_checkpoint=_none_if_blank(INITIAL_CHECKPOINT_PATH),
    final_checkpoint=_none_if_blank(FINAL_CHECKPOINT_PATH),
    output_dir=_none_if_blank(ANGULAR_OUTPUT_DIR),
    angular_nulls=int(ANGULAR_N_NULL),
    entry_nulls=int(ANGULAR_N_ENTRY_NULL),
    min_tail=int(ANGULAR_MIN_TAIL),
    null_seed=int(ANGULAR_NULL_SEED),
    show_plots=_as_bool(ANGULAR_SHOW_PLOTS),
)
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)
print(CONFIG)
print("BEST_CHECKPOINT_PATH =", BEST_CHECKPOINT_PATH or "<RUN_DIR>/checkpoint_best.pt")
print("ANGULAR_ENDPOINT_TOL =", ANGULAR_ENDPOINT_TOL)


In [ ]:
from IPython.display import display

RESULTS, MANIFEST = run_three_checkpoint_analysis(CONFIG)

print("Checkpoints used:")
for state, path in MANIFEST["checkpoints"].items():
    print(f"  {state:7s} step={MANIFEST['steps'][state]:7d}  {path}")
if MANIFEST["best_equals_final_step"]:
    print("NOTE: best and final have the same step; best->final may be trivial.")

display(RESULTS)
print("\nSummary CSV:", MANIFEST["summary_csv"])
print("Output directory:", MANIFEST["output_dir"])


## How to interpret the output

For each matrix and angular sector, compare `initial->best`, `initial->final`, and `best->final`. The main numerical fields are:

- `actual_alpha`, `actual_xmin`, `actual_D`, `actual_tail_n`, `actual_tail_decades`;
- `actual_endpoint_atoms` and `actual_zero_atoms`;
- the matched-null intervals `null_alpha_*`, `null_xmin_*`, `null_D_*`, `null_tail_n_*`, `null_tail_decades_*`;
- `full_continuous_ks_mc_p` — is the entire continuous angular spectrum distinguishable from random?;
- `tail_conditional_ks_mc_p` — above the trained package-selected `xmin`, is the far-tail distribution distinguishable from random?;
- `candidate_nonrandom_long_tail` — conservative flag requiring a nonrandom far-tail KS result and a tail longer than the random 97.5% null limit.

The key figures are the package-native linear/log PDF, CDF and CCDF plots, the pairwise far-tail CCDF zooms against the random 95% envelope, and the pairwise comparison plots for alpha, tail decades, KS $D$, and $x_{min}$.

For square attention matrices the tilt sector is structurally trivial or nearly so; twist is usually the informative angular sector. A visually straight log-log segment or an $\alpha$ near 2 is **not** sufficient evidence because random angular geometry can itself generate apparent power-law behavior.
